# 🎬 Netflix Dataset — ETL Tutorial & Practice

**Dataset:** [Kaggle — Netflix Movies and TV Shows](https://www.kaggle.com/datasets/shivamb/netflix-shows)  
**File:** `netflix_titles.csv`

This notebook is split into two parts:

| Part | What you'll do |
|------|---------------|
| **Part 1 — Tutorial** | Learn each ETL concept with clear explanations and runnable examples |
| **Part 2 — Practice** | Apply every concept to the real Netflix dataset. Each question hides a 💡 tip when you need it |

---


## ⚙️ Setup

Run this cell first every time you open the notebook.

In [ ]:
import pandas as pd
import json
from pathlib import Path

# ── point this at your downloaded CSV ──────────────────────────────────────
CSV_PATH = Path("netflix_titles.csv")
OUT_DIR  = Path("output")
OUT_DIR.mkdir(exist_ok=True)

print("pandas:", pd.__version__)
print("CSV found:", CSV_PATH.exists())


---
# Part 1 — Tutorial

Work through each section top-to-bottom. Every concept builds on the previous one.


## 1 · Read a CSV

`pd.read_csv()` loads a comma-separated file into a **DataFrame** — a table of rows and columns.

### Key argument: `dtype=str`
By default pandas *guesses* column types, which can silently corrupt values (e.g. leading zeros in IDs, ambiguous dates).  
Passing `dtype=str` loads everything as plain text so **you** decide what each column becomes.

```python
df = pd.read_csv("file.csv", dtype=str)
```

### Useful inspection methods

| Method | What it shows |
|--------|--------------|
| `df.shape` | `(rows, columns)` |
| `df.columns.tolist()` | list of column names |
| `df.head(n)` | first *n* rows |
| `df.dtypes` | current type of each column |
| `df.isna().sum()` | count of missing values per column |


In [ ]:
# ── Example: reading a tiny in-memory CSV ─────────────────────────────────
import io

sample = """id,title,year,rating
1,Inception,2010,8.8
2,The Crown,,9.1
3,Stranger Things,2016,
"""

df_example = pd.read_csv(io.StringIO(sample), dtype=str)

print("Shape:", df_example.shape)
print()
print(df_example)
print()
print("Dtypes:\n", df_example.dtypes)
print()
print("Missing values:\n", df_example.isna().sum())


---
## 2 · Rename Columns

Raw data often has cryptic names. `df.rename(columns={old: new, ...})` swaps them out.

```python
df = df.rename(columns={
    "listed_in": "genres",
    "rating":    "maturity_rating",
})
```

> **Tip:** always reassign the result — `rename` returns a new DataFrame by default.


In [ ]:
# ── Example ───────────────────────────────────────────────────────────────
df_renamed = df_example.rename(columns={
    "id":     "show_id",
    "rating": "score",
})

print("Before:", df_example.columns.tolist())
print("After: ", df_renamed.columns.tolist())


---
## 3 · Fill Missing Values (NAs)

Missing cells are represented as `NaN` (Not a Number). You must handle them **before** casting types, otherwise the cast either fails or propagates `NaN` unexpectedly.

```python
df["director"] = df["director"].fillna("Unknown")   # fill with a string
df["score"]    = df["score"].fillna(0)               # fill with a number
```

### When to fill vs. drop

| Situation | What to do |
|-----------|-----------|
| Column is sparse but needed | fill with a sentinel (`"Unknown"`, `""`, `0`) |
| Row is entirely empty | `df.dropna(how="all")` |
| Column is >50% missing and unneeded | `df.drop(columns=["col"])` |


In [ ]:
# ── Example ───────────────────────────────────────────────────────────────
print("Before fill:\n", df_example.isna().sum())

df_filled = df_example.copy()
df_filled["title"]  = df_filled["title"].fillna("Untitled")
df_filled["year"]   = df_filled["year"].fillna("0")
df_filled["rating"] = df_filled["rating"].fillna("0.0")

print("\nAfter fill:\n", df_filled.isna().sum())
print()
print(df_filled)


---
## 4 · Cast Types

After filling NAs, convert columns to the types that reflect what the data really is.

### Common casts

| Target type | Code | Use when |
|-------------|------|----------|
| Integer (no NaN) | `df["col"].astype(int)` | whole numbers, guaranteed non-null |
| Nullable integer | `pd.to_numeric(...).astype("Int32")` | whole numbers, may have NaN |
| Float | `pd.to_numeric(..., errors="coerce")` | decimals, coerce bad values to NaN |
| Datetime | `pd.to_datetime(..., errors="coerce")` | dates / timestamps |
| Categorical | `df["col"].astype("category")` | few unique string values |
| List of strings | `df["col"].apply(lambda x: x.split(","))` | comma-separated values in one cell |

> `errors="coerce"` turns un-parseable values into `NaN`/`NaT` instead of raising an exception — usually the safest choice on real data.


In [ ]:
# ── Example ───────────────────────────────────────────────────────────────
df_typed = df_filled.copy()

# integer (nullable so it handles the 0 we filled)
df_typed["year"] = pd.to_numeric(df_typed["year"], errors="coerce").astype("Int32")

# float
df_typed["rating"] = pd.to_numeric(df_typed["rating"], errors="coerce")

# category
df_typed["show_id"] = df_typed["show_id"].astype("category")

print(df_typed.dtypes)
print()
print(df_typed)


---
## 5 · Write to JSON

`df.to_json(path, orient="records")` writes one JSON object per row.

```python
df.to_json("out.json", orient="records", indent=2, force_ascii=False)
```

### Before writing: convert pandas-specific types
JSON doesn't know about `datetime64`, `Int32`, or `Categorical`. Convert them first:

```python
records = df.copy()
records["date"] = records["date"].apply(lambda v: v.isoformat() if pd.notna(v) else None)
records["year"] = records["year"].apply(lambda v: None if pd.isna(v) else int(v))
records["type"] = records["type"].astype(str)   # Categorical → str
```


In [ ]:
# ── Example ───────────────────────────────────────────────────────────────
import tempfile, json

tmp = Path(tempfile.mktemp(suffix=".json"))

records = df_typed.copy()
records["year"]   = records["year"].apply(lambda v: None if pd.isna(v) else int(v))
records["rating"] = records["rating"].apply(lambda v: None if pd.isna(v) else round(v, 1))

records.to_json(tmp, orient="records", indent=2, force_ascii=False)

# peek at the output
with open(tmp) as f:
    print(f.read())


---
## 6 · Write to Parquet

Parquet is a **columnar binary format** — far smaller and faster to query than CSV or JSON.

```python
pip install pyarrow   # required engine
df.to_parquet("out.parquet", index=False, engine="pyarrow")
```

### Before writing: handle non-native types
Parquet (via PyArrow) doesn't store Python `list` objects or pandas `Categorical` directly without a schema. Convert them first:

```python
pq_df["genres"]       = pq_df["genres"].apply(json.dumps)   # list → JSON string
pq_df["content_type"] = pq_df["content_type"].astype(str)   # Categorical → str
```

### Reading back

```python
df_back = pd.read_parquet("out.parquet", engine="pyarrow")
```


In [ ]:
# ── Example ───────────────────────────────────────────────────────────────
import tempfile

tmp_pq = Path(tempfile.mktemp(suffix=".parquet"))

pq = df_typed.copy()
pq["show_id"] = pq["show_id"].astype(str)   # Categorical → str

pq.to_parquet(tmp_pq, index=False, engine="pyarrow")

# read it back
df_back = pd.read_parquet(tmp_pq, engine="pyarrow")
print("Rows reloaded:", len(df_back))
print(df_back)


---
# Part 2 — Practice with Netflix Data

Now apply everything to the real dataset. Work through each question on your own first, then expand the **💡 Tip** if you get stuck.

> Make sure you ran the **Setup** cell at the top before starting here.


---
## Question 1 · Read the Netflix CSV

Load `netflix_titles.csv` into a DataFrame called `df`.  
Then print:
- How many rows and columns it has  
- All 12 column names  
- The first 3 rows  
- How many missing values each column has  

<details>
<summary>💡 Tip — click to reveal</summary>

```python
df = pd.read_csv(CSV_PATH, dtype=str)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head(3))
print("\nMissing values:\n", df.isna().sum())
```
</details>


In [ ]:
# ── Your answer ───────────────────────────────────────────────────────────





---
## Question 2 · Rename the Columns

Using the mapping below, rename all 12 columns and confirm the new names with `df.columns.tolist()`.

| Original | Renamed |
|----------|---------|
| `show_id` | `id` |
| `type` | `content_type` |
| `cast` | `cast_members` |
| `rating` | `maturity_rating` |
| `listed_in` | `genres` |

All other column names stay the same.

<details>
<summary>💡 Tip — click to reveal</summary>

```python
COLUMN_RENAMES = {
    "show_id":      "id",
    "type":         "content_type",
    "title":        "title",
    "director":     "director",
    "cast":         "cast_members",
    "country":      "country",
    "date_added":   "date_added",
    "release_year": "release_year",
    "rating":       "maturity_rating",
    "duration":     "duration",
    "listed_in":    "genres",
    "description":  "description",
}

df = df.rename(columns=COLUMN_RENAMES)
print(df.columns.tolist())
```
</details>


In [ ]:
# ── Your answer ───────────────────────────────────────────────────────────





---
## Question 3 · Fill Missing Values

Fill missing values in these columns using the sentinels below, then verify no NAs remain in them.

| Column | Fill with |
|--------|-----------|
| `director` | `"Unknown"` |
| `cast_members` | `"Unknown"` |
| `country` | `"Unknown"` |
| `date_added` | `""` (empty string) |
| `maturity_rating` | `"NR"` |
| `duration` | `""` (empty string) |

<details>
<summary>💡 Tip — click to reveal</summary>

```python
FILL_NA = {
    "director":        "Unknown",
    "cast_members":    "Unknown",
    "country":         "Unknown",
    "date_added":      "",
    "maturity_rating": "NR",
    "duration":        "",
}

for col, fill in FILL_NA.items():
    df[col] = df[col].fillna(fill)

# verify
cols_to_check = list(FILL_NA.keys())
print(df[cols_to_check].isna().sum())
```
</details>


In [ ]:
# ── Your answer ───────────────────────────────────────────────────────────





---
## Question 4 · Cast to the Right Types

Apply these four type casts, then print `df.dtypes` to confirm.

| Column | Target type | Notes |
|--------|-------------|-------|
| `release_year` | nullable integer `Int32` | use `pd.to_numeric(..., errors="coerce")` |
| `date_added` | datetime | format is `"%B %d, %Y"` e.g. `"September 25, 2021"` |
| `content_type` | `category` | only two values: Movie / TV Show |
| `genres` | list of strings | split on `","` and strip whitespace |

<details>
<summary>💡 Tip — click to reveal</summary>

```python
# release_year → nullable integer
df["release_year"] = pd.to_numeric(df["release_year"], errors="coerce").astype("Int32")

# date_added → datetime
df["date_added"] = pd.to_datetime(
    df["date_added"].str.strip(),
    format="%B %d, %Y",
    errors="coerce",
)

# content_type → Categorical
df["content_type"] = df["content_type"].astype("category")

# genres → list of strings
df["genres"] = df["genres"].apply(
    lambda x: [g.strip() for g in x.split(",")] if isinstance(x, str) and x else []
)

print(df.dtypes)
```
</details>


In [ ]:
# ── Your answer ───────────────────────────────────────────────────────────





---
## Question 5 · Write to JSON

Write the transformed DataFrame to `output/netflix_titles.json`.

Before writing, remember to convert:
- `date_added` → ISO 8601 string (or `None` if `NaT`)
- `release_year` → plain `int` (or `None` if `NA`)
- `content_type` → plain `str` (it's currently Categorical)

After writing, reload the file and print the **first record** to confirm it looks right.

<details>
<summary>💡 Tip — click to reveal</summary>

```python
json_path = OUT_DIR / "netflix_titles.json"

records = df.copy()
records["date_added"]   = records["date_added"].apply(
    lambda v: v.isoformat() if pd.notna(v) else None
)
records["content_type"] = records["content_type"].astype(str)
records["release_year"] = records["release_year"].apply(
    lambda v: None if pd.isna(v) else int(v)
)

records.to_json(json_path, orient="records", indent=2, force_ascii=False)
print(f"Written → {json_path}  ({json_path.stat().st_size / 1024:.1f} KB)")

# peek
with open(json_path) as f:
    sample = json.load(f)
print(sample[0])
```
</details>


In [ ]:
# ── Your answer ───────────────────────────────────────────────────────────





---
## Question 6 · Write to Parquet

Write the transformed DataFrame to `output/netflix_titles.parquet`.

Before writing, convert:
- `content_type` → `str` (Categorical → plain string)
- `genres` → JSON-encoded string (PyArrow can't store Python lists without a schema)

After writing, reload and print the shape to confirm nothing was lost.

<details>
<summary>💡 Tip — click to reveal</summary>

```python
parquet_path = OUT_DIR / "netflix_titles.parquet"

pq_df = df.copy()
pq_df["content_type"] = pq_df["content_type"].astype(str)
pq_df["genres"]       = pq_df["genres"].apply(json.dumps)

pq_df.to_parquet(parquet_path, index=False, engine="pyarrow")
print(f"Written → {parquet_path}  ({parquet_path.stat().st_size / 1024:.1f} KB)")

# reload
df_back = pd.read_parquet(parquet_path, engine="pyarrow")
print("Reloaded shape:", df_back.shape)
print(df_back.head(2))
```
</details>


In [ ]:
# ── Your answer ───────────────────────────────────────────────────────────





---
## Question 7 · Verify All Three Files Match

Reload both output files and confirm:

1. All three (CSV DataFrame, JSON, Parquet) have the **same number of rows**
2. All three have the **same column names**
3. The **first title** is identical across all three

Print a summary table and a final ✅ message if everything passes.

<details>
<summary>💡 Tip — click to reveal</summary>

```python
json_path    = OUT_DIR / "netflix_titles.json"
parquet_path = OUT_DIR / "netflix_titles.parquet"

df_json = pd.read_json(json_path, orient="records")
df_pq   = pd.read_parquet(parquet_path, engine="pyarrow")

checks = {
    "CSV rows":        len(df),
    "JSON rows":       len(df_json),
    "Parquet rows":    len(df_pq),
    "CSV columns":     len(df.columns),
    "JSON columns":    len(df_json.columns),
    "Parquet columns": len(df_pq.columns),
}
for label, val in checks.items():
    print(f"  {label:<18} {val:,}")

assert len(df) == len(df_json) == len(df_pq), "Row count mismatch!"
assert set(df.columns) == set(df_json.columns) == set(df_pq.columns), "Column mismatch!"

t_csv  = df["title"].iloc[0]
t_json = df_json["title"].iloc[0]
t_pq   = df_pq["title"].iloc[0]
assert t_csv == t_json == t_pq, f"Title mismatch: {t_csv!r} vs {t_json!r} vs {t_pq!r}"

print(f"\n✅ All checks passed.  First title: {t_csv!r}")
```
</details>


In [ ]:
# ── Your answer ───────────────────────────────────────────────────────────





---
## 🏆 Bonus Challenges

Finished the main questions? Try these to go further.

**Bonus 1 — Filter before writing**  
Write a separate JSON file that contains **only Movies** released after 2015.

**Bonus 2 — Explode genres**  
Use `df.explode("genres")` on the genres list column to get one row per genre, then count how many titles belong to each genre with `value_counts()`.

**Bonus 3 — File size comparison**  
Compare the size in KB of the original CSV, the JSON output, and the Parquet output.  
Which format is smallest? Why do you think that is?

**Bonus 4 — Round-trip test**  
For Parquet, the `genres` column was stored as a JSON string. Write a function that converts it back to a Python list after reading, so the data is identical to the in-memory DataFrame.


In [ ]:
# ── Bonus workspace ───────────────────────────────────────────────────────



